# 次単語予測 (Next Word Prediction) with PyTorch

入力した文章に続く次の単語を予測するモデルを作る。

## 全体の流れ
1. **データ準備** — テキストをトークンに分割し、数値IDに変換する
2. **Dataset / DataLoader** — PyTorch の学習ループに流せる形にする
3. **モデル定義** — Embedding + LSTM で次単語を予測する
4. **学習ループ** — loss を計算して重みを更新する
5. **推論** — 入力文に続く単語を予測する

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Step 1: データ準備

生のテキストを、モデルが扱える数値の列に変換する。

### やること
- テキストを単語に分割（トークン化）
- 全単語の一覧（vocab）を作り、各単語に数値IDを割り当てる
- `word → id` と `id → word` の辞書を作る

In [ ]:
# 学習に使うテキスト（好きな文章に変えてOK）
text = """
the cat sat on the mat
the dog sat on the log
the cat and the dog are friends
the mat is on the floor
"""

# TODO: テキストを単語のリストに分割してみよう
# ヒント: str.split() を使うと空白で分割できる
tokens = None  # <- ここを実装

print("トークン数:", len(tokens))
print("最初の10トークン:", tokens[:10])

In [ ]:
# TODO: vocab（単語の一覧）を作り、word_to_id と id_to_word の辞書を作ろう
# ヒント: set() で重複を除いた単語一覧が作れる
# ヒント: enumerate() で (index, word) のペアが取れる

vocab = None          # 重複なしの単語セット
word_to_id = {}       # 例: {"the": 0, "cat": 1, ...}
id_to_word = {}       # 例: {0: "the", 1: "cat", ...}

# TODO: tokens を ID の列に変換しよう
token_ids = None      # 例: [0, 1, 2, 3, 0, 4, ...]

print("語彙数:", len(vocab))
print("token_ids の最初の10個:", token_ids[:10])

## Step 2: Dataset / DataLoader

PyTorch でデータを扱うには `Dataset` クラスを作る必要がある。

### 考え方
テキスト `[w0, w1, w2, w3, w4, ...]` から、こういうペアを作る：

| 入力 (x)              | 正解 (y) |
|-----------------------|----------|
| `[w0, w1, w2]`        | `w3`     |
| `[w1, w2, w3]`        | `w4`     |
| `[w2, w3, w4]`        | `w5`     |

`context_len` = 入力として使う単語数（ウィンドウサイズ）

In [ ]:
context_len = 3  # 何単語を入力として使うか

class NextWordDataset(Dataset):
    def __init__(self, token_ids, context_len):
        self.data = []
        # TODO: (x, y) のペアを self.data に追加しよう
        # ヒント: token_ids を context_len 分ずつスライドしながら切り出す
        # ヒント: x = token_ids[i : i + context_len]  （長さ context_len のリスト）
        #         y = token_ids[i + context_len]       （次の1単語）
        # ヒント: range(len(token_ids) - context_len) でループできる
        pass

    def __len__(self):
        # TODO: データ数を返す
        pass

    def __getitem__(self, idx):
        x, y = self.data[idx]
        # TODO: x と y を torch.tensor に変換して返す
        # ヒント: x は torch.long 型の1次元テンソル、y はスカラーのテンソル
        pass


dataset = NextWordDataset(token_ids, context_len)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print("データ数:", len(dataset))
x_sample, y_sample = dataset[0]
print("x の形:", x_sample.shape, "→", x_sample)
print("y:", y_sample)

## Step 3: モデル定義

単語ID → Embedding → LSTM → Linear → 次単語の確率

```
入力 [3, 0, 1]  (単語IDの列)
  ↓  nn.Embedding   # 各IDをベクトルに変換
  ↓  nn.LSTM        # 系列を処理して文脈を捉える
  ↓  nn.Linear      # vocab_size 次元に射影
出力 [0.1, 0.3, 0.05, ...]  (各単語のスコア)
```

In [ ]:
class NextWordModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # TODO: 以下の3層を定義しよう
        # self.embedding = nn.Embedding(???, ???)
        #   vocab_size 個の単語を、embed_dim 次元のベクトルに変換する
        # self.lstm = nn.LSTM(???, ???, batch_first=True)
        #   入力次元=embed_dim、隠れ層次元=hidden_dim
        # self.fc = nn.Linear(???, ???)
        #   hidden_dim から vocab_size 次元に変換する（最終出力）
        pass

    def forward(self, x):
        # x の形: (batch_size, context_len)  ← 単語IDの列
        # TODO: 順番に層を通していこう
        # 1. self.embedding(x)  → (batch, context_len, embed_dim)
        # 2. self.lstm(embedded) → output は (batch, context_len, hidden_dim)
        #    ヒント: lstm は (output, (h_n, c_n)) のタプルを返す
        # 3. 最後のタイムステップだけ取り出す: output[:, -1, :]
        # 4. self.fc(...) で (batch, vocab_size) に変換して return
        pass


# ハイパーパラメータ（自由に変えてみよう）
embed_dim  = 16
hidden_dim = 32
vocab_size = len(vocab)

model = NextWordModel(vocab_size, embed_dim, hidden_dim)
print(model)

## Step 4: 学習ループ

1. モデルに x を渡して出力（logits）を得る
2. logits と正解 y を比べて loss を計算する
3. `loss.backward()` で勾配を計算する
4. `optimizer.step()` で重みを更新する
5. これを epoch 回繰り返す

In [ ]:
num_epochs = 100
learning_rate = 0.01

# 損失関数とオプティマイザ
# CrossEntropyLoss = 多クラス分類の定番。softmax も内包している
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    total_loss = 0.0

    for x_batch, y_batch in dataloader:
        # TODO: 学習ステップを実装しよう
        # 1. optimizer.zero_grad()  ← 勾配をリセット（忘れずに！）
        # 2. logits = model(x_batch)
        # 3. loss = criterion(logits, y_batch)
        # 4. loss.backward()
        # 5. optimizer.step()
        # 6. total_loss += loss.item()
        pass

    if (epoch + 1) % 10 == 0:
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1:3d} / {num_epochs}  loss: {avg_loss:.4f}")

## Step 5: 推論

学習済みモデルに文章を渡して、次の単語を予測する。

In [ ]:
def predict_next_word(prompt: str, model, word_to_id, id_to_word, context_len):
    model.eval()  # 推論モードに切り替え（dropout等が無効になる）

    # TODO: prompt を単語IDの列に変換しよう
    # ヒント: prompt.split() でトークン化し、word_to_id で ID に変換する
    # ヒント: 最後の context_len 個だけを使う（[-context_len:]）
    words = None  # <- ここを実装
    ids   = None  # <- ここを実装

    # TODO: ids を (1, context_len) 形状の LongTensor にして model に渡そう
    # ヒント: torch.tensor([ids]) で形状 (1, context_len) になる
    with torch.no_grad():
        logits = None  # <- ここを実装

    # TODO: 最も確率が高い単語IDを取り出し、id_to_word で単語に変換しよう
    # ヒント: torch.argmax(logits, dim=-1) でスコアが最大のインデックスが得られる
    predicted_id   = None  # <- ここを実装
    predicted_word = None  # <- ここを実装

    return predicted_word


# 動作確認
prompt = "the cat sat"
result = predict_next_word(prompt, model, word_to_id, id_to_word, context_len)
print(f"入力: '{prompt}'  →  次の単語: '{result}'")